In [1]:
import pandas as pd
import requests

In [2]:
url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=20000"

In [3]:
response = requests.get(url)

In [4]:
response.status_code

200

In [5]:
data=response.json()

In [6]:
data[1][0]

{'indicator': {'id': 'NY.GDP.PCAP.CD',
  'value': 'GDP per capita (current US$)'},
 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'},
 'countryiso3code': 'AFE',
 'date': '2025',
 'value': 1722.38561960923,
 'unit': '',
 'obs_status': '',
 'decimal': 1}

In [8]:
gdp = pd.DataFrame(data[1])

In [11]:
gdp = gdp.rename(columns={
    "country": "Country",
    "countryiso3code": "Country_Code",
    "date": "Year",
    "value": "GDP_per_capita"
})

In [13]:
gdp["Country"] = gdp["Country"].apply(lambda x: x["value"])

In [16]:
gdp["Year"]=pd.to_numeric(gdp["Year"])

In [17]:
gdp["GDP_per_capita"].isnull().sum()

np.int64(2745)

In [19]:
gdp.describe()

,Year,GDP_per_capita,decimal
count,17490.000000,14745.000000,17490.0
mean,1992.500000,8919.926659,1.0
std,19.050916,17945.875543,0.0
min,1960.000000,11.801322,1.0
25%,1976.000000,595.918898,1.0
50%,1992.500000,1993.586672,1.0
75%,2009.000000,8243.898955,1.0
max,2025.000000,288001.574856,1.0


In [20]:
gdp[gdp["Country"] == "India"][["Country", "Year", "GDP_per_capita"]]

,Country,Year,GDP_per_capita
9042,India,2025,2702.479871
9043,India,2024,2591.991661
9044,India,2023,2434.448263
9045,India,2022,2279.981457
9046,India,2021,2239.613844
...,...,...,...
9103,India,1964,117.856431
9104,India,1963,103.435021
9105,India,1962,92.199958
9106,India,1961,87.853861


In [21]:
gdp_clean = gdp.dropna(subset=["GDP_per_capita"]).copy()

In [22]:
gdp_clean.shape

(14745, 8)

In [25]:
gdp_analysis = gdp_clean[
    ["Country", "Country_Code", "Year", "GDP_per_capita"]
].copy()

In [26]:
gdp_analysis.head()

,Country,Country_Code,Year,GDP_per_capita
0,Africa Eastern and Southern,AFE,2025,1722.385620
1,Africa Eastern and Southern,AFE,2024,1628.227289
2,Africa Eastern and Southern,AFE,2023,1571.132704
3,Africa Eastern and Southern,AFE,2022,1675.902524
4,Africa Eastern and Southern,AFE,2021,1560.894626


In [27]:
gdp_analysis.duplicated(
    subset=["Country_Code", "Year"]
).sum()

np.int64(198)

In [28]:
duplicates = gdp_analysis[
    gdp_analysis.duplicated(
        subset=["Country_Code", "Year"],
        keep=False
    )
].sort_values(["Country_Code", "Year"])

duplicates.head(30)

,Country,Country_Code,Year,GDP_per_capita
1055,High income,,1960,1205.228067
1847,Low income,,1960,107.153557
1913,Lower middle income,,1960,108.275586
3101,Upper middle income,,1960,147.175575
1054,High income,,1961,1269.994821
1846,Low income,,1961,106.031172
1912,Lower middle income,,1961,112.326607
3100,Upper middle income,,1961,144.912830
1053,High income,,1962,1356.633916
1845,Low income,,1962,115.741716


In [29]:
duplicates[
    duplicates.duplicated(
        subset=["Country_Code", "Year"],
        keep=False
    )
].head(50)

,Country,Country_Code,Year,GDP_per_capita
1055,High income,,1960,1205.228067
1847,Low income,,1960,107.153557
1913,Lower middle income,,1960,108.275586
3101,Upper middle income,,1960,147.175575
1054,High income,,1961,1269.994821
1846,Low income,,1961,106.031172
1912,Lower middle income,,1961,112.326607
3100,Upper middle income,,1961,144.912830
1053,High income,,1962,1356.633916
1845,Low income,,1962,115.741716


In [31]:
country_url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

In [32]:
country_response = requests.get(country_url)

In [36]:
country_data = country_response.json()

In [37]:
countries = pd.DataFrame(country_data[1])

In [38]:
countries["Region"] = countries["region"].apply(lambda x: x["value"])

In [39]:
countries[["id", "name", "Region"]].head(20)

,id,name,Region
0,ABW,Aruba,Latin America & Caribbean
1,AFE,Africa Eastern and Southern,Aggregates
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan"
3,AFR,Africa,Aggregates
4,AFW,Africa Western and Central,Aggregates
5,AGO,Angola,Sub-Saharan Africa
6,ALB,Albania,Europe & Central Asia
7,AND,Andorra,Europe & Central Asia
8,ARB,Arab World,Aggregates
9,ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan"


In [40]:
countries_only = countries[
    countries["Region"] != "Aggregates"
].copy()

In [41]:
gdp_analysis = gdp_analysis.merge(
    countries_only[["id", "Region"]],
    left_on="Country_Code",
    right_on="id",
    how="inner"
)

In [42]:
gdp_analysis = gdp_analysis.drop(columns=["id"])

In [43]:
gdp_analysis.head()

,Country,Country_Code,Year,GDP_per_capita,Region
0,Afghanistan,AFG,2024,416.871146,"Middle East, North Africa, Afghanistan & Pakistan"
1,Afghanistan,AFG,2023,413.757895,"Middle East, North Africa, Afghanistan & Pakistan"
2,Afghanistan,AFG,2022,357.261153,"Middle East, North Africa, Afghanistan & Pakistan"
3,Afghanistan,AFG,2021,356.496214,"Middle East, North Africa, Afghanistan & Pakistan"
4,Afghanistan,AFG,2020,510.787063,"Middle East, North Africa, Afghanistan & Pakistan"


In [44]:
gdp_analysis.duplicated(
    subset=["Country_Code", "Year"]
).sum()

np.int64(0)

In [45]:
gdp_analysis[
    gdp_analysis["Country"] == "India"
][["Country", "Country_Code", "Year", "GDP_per_capita", "Region"]]

,Country,Country_Code,Year,GDP_per_capita,Region
4843,India,IND,2025,2702.479871,South Asia
4844,India,IND,2024,2591.991661,South Asia
4845,India,IND,2023,2434.448263,South Asia
4846,India,IND,2022,2279.981457,South Asia
4847,India,IND,2021,2239.613844,South Asia
...,...,...,...,...,...
4904,India,IND,1964,117.856431,South Asia
4905,India,IND,1963,103.435021,South Asia
4906,India,IND,1962,92.199958,South Asia
4907,India,IND,1961,87.853861,South Asia


In [46]:
gdp_analysis.to_csv(
    "../data/gdp_per_capita_clean.csv",
    index=False
)